# IndoMeme-XAI Revision V3 — XAI Representative ICAM

## Frozen Representative C — Seed 999

Notebook ini menjalankan tahap XAI setelah predictive experiment dan official-test evaluation selesai.

**Scientific constraints:**

- architecture: `C` / ICAM V3;
- representative seed ditentukan **sebelum melihat official test**;
- representative checkpoint dan threshold dibaca dari frozen model-decision record;
- tidak ada training;
- tidak ada threshold calibration;
- tidak ada model/seed selection;
- tidak ada pemilihan sample berdasarkan kualitas explanation;
- seluruh 1.005 official-test samples diproses;
- output explanation per-sample dipertahankan untuk perturbation dan case-study tahap berikutnya.

Frozen XAI protocol:

1. Integrated Gradients — 50 steps, target `predicted-class logit`, semantic text tokens;
2. actual cross-attention `text CLS → 196 image patches`, per-head lalu mean;
3. ViT attention rollout dengan residual identity;
4. internal ICAM incongruity weights `a_j`.

Untuk ranking text pada faithfulness tahap berikutnya, skor yang digunakan adalah nilai absolut dari signed Integrated Gradients setelah agregasi dimensi embedding.

## 1. Imports, frozen hashes, dan artifact discovery

Tambahkan sebagai Kaggle Input:

1. frozen dataset utama;
2. final Variant C checkpoint output;
3. `07_v3_model_decision_freeze_v2` output;
4. `08_v3_final_official_test_v2` output.

Notebook akan mencari freeze/test artifact secara rekursif di `/kaggle/input`.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json
import math
import os
import random
import re
import time
import unicodedata

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import yaml

EXPECTED_PROTOCOL_SHA256 = (
    "d3ba1de81ef89a67946c43af996396d294d35f01a1640168bab18c8a3bec4f12"
)

EXPECTED_FREEZE_SHA256 = (
    "63f8bba319f07928f3b0528bd3e8ebc3b7c325e77e3d3db01e7da6119ce31f62"
)

EXPECTED_FINAL_TEST_RECORD_SHA256 = (
    "c67b664e1ecb07bbbf0243550862e04ce3d2583518b18158c66c3ccc9fd5676b"
)

EXPECTED_REPRESENTATIVE_VARIANT = "C"
EXPECTED_REPRESENTATIVE_SEED = 999
EXPECTED_REPRESENTATIVE_CHECKPOINT_SHA256 = (
    "fc76e1b4bf2288e99802356d63a8ad5345cef54732767c25c9e51178d21520c4"
)

PROTOCOL_ROOT = Path(
    "/kaggle/input/datasets/reynaldaryansyah/dataset"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/reynaldaryansyah/dataset/"
    "indomeme_xai_revision/indomeme_xai_revision"
)

PROTOCOL_YAML = PROTOCOL_ROOT / "protocol_v3.yaml"
TEST_MANIFEST = DATA_ROOT / "manifests" / "test_manifest.csv"
IMAGE_ROOT = DATA_ROOT / "source_snapshot" / "images"

OUTPUT_ROOT = Path(
    "/kaggle/working/indomeme_xai_revision_v3/"
    "xai_representative_C_seed999"
)

SHARD_ROOT = OUTPUT_ROOT / "shards"

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SHARD_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

if DEVICE.type != "cuda":
    raise RuntimeError(
        "XAI representative C membutuhkan CUDA GPU."
    )

# Execution-only settings. These do not alter the frozen scientific method.
XAI_BATCH_SIZE = 2
SHARD_SIZE = 25
IG_STEPS = 50
IG_STEP_CHUNK = 10

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("DEVICE:", DEVICE)
print("IG_STEPS:", IG_STEPS)
print("XAI_BATCH_SIZE:", XAI_BATCH_SIZE)
print("SHARD_SIZE:", SHARD_SIZE)

OUTPUT_ROOT: /kaggle/working/indomeme_xai_revision_v3/xai_representative_C_seed999
DEVICE: cuda:0
IG_STEPS: 50
XAI_BATCH_SIZE: 2
SHARD_SIZE: 25


## 2. Verify frozen protocol, model-decision freeze, dan final official-test record

In [2]:
actual_protocol_sha = sha256_file(
    PROTOCOL_YAML
)

if (
    actual_protocol_sha
    != EXPECTED_PROTOCOL_SHA256
):
    raise RuntimeError(
        "Frozen protocol SHA mismatch."
    )

protocol = yaml.safe_load(
    PROTOCOL_YAML.read_text(
        encoding="utf-8"
    )
)

if (
    protocol["protocol"]["status"]
    != "FROZEN"
):
    raise RuntimeError(
        "Protocol status bukan FROZEN."
    )

if (
    int(
        protocol["xai"]
        ["integrated_gradients"]
        ["steps"]
    )
    != IG_STEPS
):
    raise RuntimeError(
        "Frozen IG step count mismatch."
    )

if (
    protocol["xai"]
    ["integrated_gradients"]
    ["target"]
    != "predicted-class logit"
):
    raise RuntimeError(
        "Frozen IG target mismatch."
    )

freeze_candidates = sorted(
    Path("/kaggle/input").rglob(
        "model_decision_freeze_record.json"
    )
)

if len(
    freeze_candidates
) != 1:
    raise RuntimeError(
        "Harus ada tepat satu model_decision_freeze_record.json. "
        f"Ditemukan {len(freeze_candidates)}."
    )

FREEZE_RECORD_PATH = freeze_candidates[
    0
]

actual_freeze_sha = sha256_file(
    FREEZE_RECORD_PATH
)

if (
    actual_freeze_sha
    != EXPECTED_FREEZE_SHA256
):
    raise RuntimeError(
        "Model-decision freeze SHA mismatch."
    )

freeze_record = json.loads(
    FREEZE_RECORD_PATH.read_text(
        encoding="utf-8"
    )
)

final_test_candidates = sorted(
    Path("/kaggle/input").rglob(
        "final_official_test_record.json"
    )
)

if len(
    final_test_candidates
) != 1:
    raise RuntimeError(
        "Harus ada tepat satu final_official_test_record.json. "
        f"Ditemukan {len(final_test_candidates)}."
    )

FINAL_TEST_RECORD_PATH = final_test_candidates[
    0
]

actual_final_test_sha = sha256_file(
    FINAL_TEST_RECORD_PATH
)

if (
    actual_final_test_sha
    != EXPECTED_FINAL_TEST_RECORD_SHA256
):
    raise RuntimeError(
        "Final official-test record SHA mismatch."
    )

final_test_record = json.loads(
    FINAL_TEST_RECORD_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    final_test_record[
        "status"
    ]
    != "FINAL_OFFICIAL_TEST_COMPLETE"
):
    raise RuntimeError(
        "Final official-test record belum complete."
    )

if final_test_record[
    "selection_or_tuning_on_test"
]:
    raise RuntimeError(
        "Invalid final-test record: test selection detected."
    )

if final_test_record[
    "threshold_calibration_on_test"
]:
    raise RuntimeError(
        "Invalid final-test record: test threshold calibration detected."
    )

representative = freeze_record[
    "representative_C"
]

if representative[
    "variant"
] != EXPECTED_REPRESENTATIVE_VARIANT:
    raise RuntimeError(
        "Representative variant mismatch."
    )

if int(
    representative[
        "selected_seed"
    ]
) != EXPECTED_REPRESENTATIVE_SEED:
    raise RuntimeError(
        "Representative seed mismatch."
    )

if representative[
    "checkpoint_sha256"
] != EXPECTED_REPRESENTATIVE_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Representative checkpoint SHA mismatch."
    )

if representative[
    "official_test_used"
] is not False:
    raise RuntimeError(
        "Representative seed must be validation-only selected."
    )

REPRESENTATIVE_THRESHOLD = float(
    representative[
        "selected_threshold"
    ]
)

print(
    json.dumps(
        {
            "protocol_sha256": actual_protocol_sha,
            "freeze_record_sha256": actual_freeze_sha,
            "final_test_record_sha256": actual_final_test_sha,
            "representative_variant": representative["variant"],
            "representative_seed": representative["selected_seed"],
            "representative_threshold": REPRESENTATIVE_THRESHOLD,
            "representative_checkpoint_sha256": representative["checkpoint_sha256"],
            "status": "PASS",
        },
        indent=2,
    )
)

{
  "protocol_sha256": "d3ba1de81ef89a67946c43af996396d294d35f01a1640168bab18c8a3bec4f12",
  "freeze_record_sha256": "63f8bba319f07928f3b0528bd3e8ebc3b7c325e77e3d3db01e7da6119ce31f62",
  "final_test_record_sha256": "c67b664e1ecb07bbbf0243550862e04ce3d2583518b18158c66c3ccc9fd5676b",
  "representative_variant": "C",
  "representative_seed": 999,
  "representative_threshold": 0.45,
  "representative_checkpoint_sha256": "fc76e1b4bf2288e99802356d63a8ad5345cef54732767c25c9e51178d21520c4",
  "status": "PASS"
}


## 3. Resolve frozen representative checkpoint dan official-test predictions

Checkpoint tidak dipilih ulang. Path pada freeze record harus tersedia sebagai persistent Kaggle Input dan SHA256-nya harus sama.

In [3]:
REPRESENTATIVE_CHECKPOINT_PATH = Path(
    representative[
        "checkpoint_path"
    ]
)

if not REPRESENTATIVE_CHECKPOINT_PATH.exists():
    # The absolute freeze path can change when a notebook output is attached
    # under a different Kaggle input slug. Resolve only by the frozen SHA.
    matching_paths = []

    for candidate in Path(
        "/kaggle/input"
    ).rglob(
        "best_model.pt"
    ):
        try:
            if (
                round(
                    candidate.stat().st_size
                    / (1024 ** 3),
                    3,
                )
                != 0.564
            ):
                continue

            if (
                sha256_file(
                    candidate
                )
                == EXPECTED_REPRESENTATIVE_CHECKPOINT_SHA256
            ):
                matching_paths.append(
                    candidate
                )

        except Exception:
            continue

    if len(
        matching_paths
    ) != 1:
        raise RuntimeError(
            "Representative C checkpoint tidak dapat di-resolve secara unik. "
            f"Matches={len(matching_paths)}"
        )

    REPRESENTATIVE_CHECKPOINT_PATH = matching_paths[
        0
    ]

checkpoint_sha = sha256_file(
    REPRESENTATIVE_CHECKPOINT_PATH
)

if (
    checkpoint_sha
    != EXPECTED_REPRESENTATIVE_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Representative checkpoint SHA verification FAIL."
    )

prediction_candidates = sorted(
    Path("/kaggle/input").rglob(
        "C_seed_999_official_test.csv"
    )
)

if len(
    prediction_candidates
) != 1:
    raise RuntimeError(
        "Harus ada tepat satu frozen official-test prediction C seed 999. "
        f"Ditemukan {len(prediction_candidates)}."
    )

OFFICIAL_PREDICTION_PATH = prediction_candidates[
    0
]

print(
    "Representative checkpoint:",
    REPRESENTATIVE_CHECKPOINT_PATH,
)

print(
    "Official prediction reference:",
    OFFICIAL_PREDICTION_PATH,
)

print(
    "Checkpoint SHA verification: PASS"
)

Representative checkpoint: /kaggle/input/notebooks/reynaldaryansyah/04c-v3-variant-c-train-ipynb/indomeme_xai_revision_v3/controlled_phase_a/variant_C/seed_999/checkpoints/best_model.pt
Official prediction reference: /kaggle/input/notebooks/reynaldaryansyah/08-v3-final-official-test/indomeme_xai_revision_v3/final_official_test/predictions/C_seed_999_official_test.csv
Checkpoint SHA verification: PASS


## 4. Load official test dan deterministic preprocessing

Tidak ada sample selection. Seluruh 1.005 official-test rows diproses dalam urutan `sample_uid` yang stabil.

In [4]:
expected_test_sha = (
    protocol[
        "dataset"
    ][
        "manifest_sha256"
    ][
        "test"
    ]
)

actual_test_sha = sha256_file(
    TEST_MANIFEST
)

if (
    actual_test_sha
    != expected_test_sha
):
    raise RuntimeError(
        "Official-test manifest SHA mismatch."
    )

test_df = pd.read_csv(
    TEST_MANIFEST
)

if len(
    test_df
) != 1005:
    raise RuntimeError(
        f"Expected 1005 official-test rows, got {len(test_df)}."
    )

if test_df[
    "sample_uid"
].duplicated().any():
    raise RuntimeError(
        "Duplicate sample_uid pada official test."
    )

test_df = (
    test_df
    .sort_values(
        "sample_uid"
    )
    .reset_index(
        drop=True
    )
)

print(
    "Official test rows:",
    len(
        test_df
    ),
)

print(
    "Official test XAI scope: ALL SAMPLES"
)

Official test rows: 1005
Official test XAI scope: ALL SAMPLES


In [5]:
from huggingface_hub import HfApi
from transformers import (
    AutoTokenizer,
    AutoImageProcessor,
    AutoModel,
)

TEXT_SPEC = protocol[
    "models"
][
    "indobertweet"
]

VISION_SPEC = protocol[
    "models"
][
    "vit"
]

api = HfApi()

text_info = api.model_info(
    repo_id=TEXT_SPEC[
        "id"
    ],
    revision=TEXT_SPEC[
        "revision"
    ],
)

vision_info = api.model_info(
    repo_id=VISION_SPEC[
        "id"
    ],
    revision=VISION_SPEC[
        "revision"
    ],
)

TEXT_REVISION = text_info.sha
VISION_REVISION = vision_info.sha

if (
    TEXT_REVISION
    != TEXT_SPEC[
        "revision"
    ]
):
    raise RuntimeError(
        "Resolved IndoBERTweet revision mismatch."
    )

if not VISION_REVISION.startswith(
    VISION_SPEC[
        "revision"
    ]
):
    raise RuntimeError(
        "Resolved ViT revision mismatch."
    )

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_SPEC[
        "id"
    ],
    revision=TEXT_REVISION,
)

vit_processor = AutoImageProcessor.from_pretrained(
    VISION_SPEC[
        "id"
    ],
    revision=VISION_REVISION,
)

validation_transform = transforms.Compose([
    transforms.Resize(
        (
            224,
            224,
        ),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=vit_processor.image_mean,
        std=vit_processor.image_std,
    ),
])

print(
    "Controlled backbone revisions + deterministic image preprocessing: PASS"
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Controlled backbone revisions + deterministic image preprocessing: PASS


In [6]:
SLANG_DICT = {
    "gw":"saya","gue":"saya","gua":"saya","guwe":"saya","aq":"aku","akoh":"aku","akuh":"aku",
    "lu":"kamu","lo":"kamu","loe":"kamu","elo":"kamu","elu":"kamu","kmu":"kamu","kamuh":"kamu",
    "ane":"saya","ente":"kamu","agan":"kamu","sis":"kakak","bro":"kawan","bray":"kawan",
    "bre":"kawan","cuy":"kawan","klian":"kalian",
    "gak":"tidak","ga":"tidak","gk":"tidak","nggak":"tidak","ngga":"tidak","ngk":"tidak",
    "kgk":"tidak","tdk":"tidak","engga":"tidak","enggak":"tidak","kga":"tidak",
    "yg":"yang","dgn":"dengan","dg":"dengan","krn":"karena","krna":"karena","utk":"untuk",
    "tp":"tapi","tpi":"tapi","klo":"kalau","klau":"kalau","kl":"kalau","klu":"kalau",
    "spy":"supaya","ato":"atau","bgt":"banget","bgtt":"banget","bngt":"banget","bnr":"benar",
    "bener":"benar","emg":"memang","emang":"memang","mmg":"memang","skrg":"sekarang",
    "skrng":"sekarang","td":"tadi","tdi":"tadi","bsk":"besok","knp":"kenapa",
    "gmn":"bagaimana","gimana":"bagaimana","gmana":"bagaimana","kyk":"seperti",
    "kayak":"seperti","kek":"seperti","udh":"sudah","udah":"sudah","uda":"sudah",
    "dah":"sudah","blm":"belum","blom":"belum","lg":"lagi","lgi":"lagi","msh":"masih",
    "msi":"masih","jd":"jadi","jdi":"jadi","tau":"tahu","taw":"tahu","tw":"tahu",
    "liat":"lihat","ngeliat":"melihat","denger":"dengar","ngomong":"bicara",
    "blng":"bilang","bilg":"bilang","aja":"saja","doang":"saja","doank":"saja",
    "sih":"","deh":"","dong":"","kok":"","lah":"","loh":"","lho":"","yah":"",
    "iy":"iya","iyaa":"iya","iye":"iya","yup":"iya","yoi":"iya","org":"orang",
    "orng":"orang","orgnya":"orangnya","dr":"dari","dri":"dari","kpd":"kepada",
    "thd":"terhadap","ttg":"tentang","abis":"habis","trs":"terus","trus":"terus",
    "tmn":"teman","temen":"teman","tman":"teman","rmh":"rumah","kntr":"kantor",
    "krj":"kerja","ank":"anak","indo":"indonesia","cwo":"cowok","cwe":"cewek",
    "cwok":"cowok","cwek":"cewek","wkwk":"haha","wkwkwk":"haha","wkwkwkwk":"haha",
    "wkwkwkwkwk":"haha","hahaha":"haha","hahahaha":"haha","hehehe":"hehe",
    "hehehehe":"hehe","anjir":"anjing","anjay":"anjing","anjrit":"anjing",
    "anjr":"anjing","anying":"anjing","njir":"anjing","bgsd":"bangsat",
    "bgst":"bangsat","bjir":"anjing","bjg":"bajingan","kntl":"kontol",
    "mmk":"memek","tai":"tahi",
}

assert len(SLANG_DICT) == 149

URL_PATTERN = re.compile(
    r"(https?://\S+|www\.\S+|\S+\.(?:com|co\.id|net|org|id|tv|me)\b\S*)",
    re.IGNORECASE,
)

MENTION_PATTERN = re.compile(
    r"@\w+"
)

HASHTAG_PATTERN = re.compile(
    r"#(\w+)"
)

ALLOWED_PUNCT = re.compile(
    r"[^\w\s\.,!?\-'\"\u2019]+",
    re.UNICODE,
)

MULTI_SPACE = re.compile(
    r"\s+"
)

def safe_text(value):
    if value is None:
        return ""

    if (
        isinstance(
            value,
            float,
        )
        and pd.isna(
            value
        )
    ):
        return ""

    return str(
        value
    ).strip()

def normalize_slang(text):
    output = []

    for token in text.split():
        prefix = ""
        suffix = ""
        core = token

        while (
            core
            and not core[0].isalnum()
        ):
            prefix += core[0]
            core = core[1:]

        while (
            core
            and not core[-1].isalnum()
        ):
            suffix = (
                core[-1]
                + suffix
            )
            core = core[:-1]

        output.append(
            f"{prefix}"
            f"{SLANG_DICT.get(core.lower(), core)}"
            f"{suffix}"
        )

    return " ".join(
        output
    )

def clean_ocr(text):
    text = safe_text(
        text
    )

    if not text:
        return ""

    text = unicodedata.normalize(
        "NFKC",
        text,
    ).lower()

    text = URL_PATTERN.sub(
        " ",
        text,
    )

    text = MENTION_PATTERN.sub(
        " ",
        text,
    )

    text = HASHTAG_PATTERN.sub(
        r"\1",
        text,
    )

    text = ALLOWED_PUNCT.sub(
        " ",
        text,
    )

    text = normalize_slang(
        text
    )

    return MULTI_SPACE.sub(
        " ",
        text,
    ).strip()

TOPIC_CATEGORIES = protocol[
    "dataset"
]["topic_categories"]

TOPIC_TO_INDEX = {
    topic: index
    for index, topic in enumerate(
        TOPIC_CATEGORIES
    )
}

def topic_to_multihot(
    value,
):
    vector = np.zeros(
        8,
        dtype=np.float32,
    )

    text = (
        safe_text(
            value
        ).lower()
        or "unknown"
    )

    parts = [
        part.strip()
        for part in text.split(",")
        if part.strip()
    ]

    if not parts:
        parts = [
            "unknown"
        ]

    for part in parts:
        if part not in TOPIC_TO_INDEX:
            raise ValueError(
                f"Topic tidak dikenal: {part!r}"
            )

        vector[
            TOPIC_TO_INDEX[
                part
            ]
        ] = 1.0

    return vector

def format_controlled_text(
    row,
    with_topic,
):
    caption = safe_text(
        row["caption"]
    )

    ocr = clean_ocr(
        row["ocr"]
    )

    if with_topic:
        topic = (
            safe_text(
                row[
                    "topic_normalized"
                ]
            ).lower()
            or "unknown"
        )

        return (
            f"[{topic}] "
            f"{caption} "
            f"[SEP] "
            f"{ocr}"
        ).strip()

    return (
        f"{caption} "
        f"[SEP] "
        f"{ocr}"
    ).strip()

assert clean_ocr(
    "GW gak liat #Kamu @abc https://x.com"
) == "saya tidak lihat kamu"

print(
    "Preprocessing implementation: PASS"
)

Preprocessing implementation: PASS


In [7]:
LABEL_TO_ID = {
    "not hate": 0,
    "hate": 1,
}

MAX_LENGTH = int(
    TEXT_SPEC[
        "max_length"
    ]
)

def resolve_image_path(
    row,
):
    old_path = safe_text(
        row[
            "local_image_path"
        ]
    ).replace(
        "\\",
        "/",
    )

    filename = Path(
        old_path
    ).name

    source_split = safe_text(
        row[
            "source_split"
        ]
    )

    resolved = (
        IMAGE_ROOT
        / source_split
        / filename
    )

    if not resolved.exists():
        raise FileNotFoundError(
            resolved
        )

    return resolved

class XAITestDataset(
    Dataset
):
    def __init__(
        self,
        frame,
    ):
        self.frame = (
            frame
            .reset_index(
                drop=True
            )
            .copy()
        )

    def __len__(
        self,
    ):
        return len(
            self.frame
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.frame.iloc[
            index
        ]

        with Image.open(
            resolve_image_path(
                row
            )
        ) as image:
            pixel_values = validation_transform(
                image.convert(
                    "RGB"
                )
            )

        return {
            "sample_uid": str(
                row[
                    "sample_uid"
                ]
            ),
            "text": format_controlled_text(
                row,
                with_topic=True,
            ),
            "pixel_values": pixel_values,
            "topic_features": torch.tensor(
                topic_to_multihot(
                    row[
                        "topic_normalized"
                    ]
                ),
                dtype=torch.float32,
            ),
            "label": torch.tensor(
                LABEL_TO_ID[
                    str(
                        row[
                            "hate_final"
                        ]
                    )
                    .strip()
                    .lower()
                ],
                dtype=torch.long,
            ),
        }

def collate_xai(
    batch,
):
    tokenized = tokenizer(
        [
            item[
                "text"
            ]
            for item in batch
        ],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    return {
        "sample_uid": [
            item[
                "sample_uid"
            ]
            for item in batch
        ],
        "input_ids": tokenized[
            "input_ids"
        ],
        "attention_mask": tokenized[
            "attention_mask"
        ],
        "pixel_values": torch.stack([
            item[
                "pixel_values"
            ]
            for item in batch
        ]),
        "topic_features": torch.stack([
            item[
                "topic_features"
            ]
            for item in batch
        ]),
        "labels": torch.stack([
            item[
                "label"
            ]
            for item in batch
        ]),
    }

xai_dataset = XAITestDataset(
    test_df
)

print(
    "XAI dataset:",
    len(
        xai_dataset
    ),
    "samples"
)

XAI dataset: 1005 samples


## 5. Reconstruct frozen ICAM C architecture

In [8]:
HIDDEN_DIM = 768
NUM_HEADS = 8
ATTN_DROPOUT = 0.1
TOPIC_PROJ_DIM = 32

def set_module_trainable(
    module,
    trainable,
):
    for parameter in module.parameters():
        parameter.requires_grad = trainable

def freeze_controlled_backbones(
    text_encoder,
    vision_encoder,
):
    set_module_trainable(
        text_encoder.embeddings,
        False,
    )

    for layer in (
        text_encoder.encoder.layer[
            :2
        ]
    ):
        set_module_trainable(
            layer,
            False,
        )

    set_module_trainable(
        vision_encoder.embeddings,
        False,
    )

    for layer in (
        vision_encoder.encoder.layer[
            :2
        ]
    ):
        set_module_trainable(
            layer,
            False,
        )

def assert_freeze_policy(
    model,
):
    assert all(
        not parameter.requires_grad
        for parameter
        in model.text_encoder.embeddings.parameters()
    )

    for layer in (
        model.text_encoder.encoder.layer[
            :2
        ]
    ):
        assert all(
            not parameter.requires_grad
            for parameter
            in layer.parameters()
        )

    assert any(
        parameter.requires_grad
        for parameter
        in model.text_encoder.encoder.layer[
            2
        ].parameters()
    )

    assert all(
        not parameter.requires_grad
        for parameter
        in model.vision_encoder.embeddings.parameters()
    )

    for layer in (
        model.vision_encoder.encoder.layer[
            :2
        ]
    ):
        assert all(
            not parameter.requires_grad
            for parameter
            in layer.parameters()
        )

    assert any(
        parameter.requires_grad
        for parameter
        in model.vision_encoder.encoder.layer[
            2
        ].parameters()
    )

class ControlledMultimodalModel(
    nn.Module
):
    def __init__(
        self,
        variant,
    ):
        super().__init__()

        if variant not in {
            "A",
            "B",
            "C",
            "D",
            "C_NO_INCONGRUITY",
            "C_NO_GATE",
        }:
            raise ValueError(
                f"Variant tidak dikenal: {variant}"
            )

        self.variant = variant
        self.uses_topic = (
            variant
            in {
                "A",
                "B",
                "C",
                "C_NO_INCONGRUITY",
                "C_NO_GATE",
            }
        )

        self.text_encoder = (
            AutoModel.from_pretrained(
                TEXT_SPEC[
                    "id"
                ],
                revision=TEXT_REVISION,
            )
        )

        self.vision_encoder = (
            AutoModel.from_pretrained(
                VISION_SPEC[
                    "id"
                ],
                revision=VISION_REVISION,
            )
        )

        if int(
            self.text_encoder.config.hidden_size
        ) != HIDDEN_DIM:
            raise RuntimeError(
                "Unexpected IndoBERTweet hidden dimension."
            )

        if int(
            self.vision_encoder.config.hidden_size
        ) != HIDDEN_DIM:
            raise RuntimeError(
                "Unexpected ViT hidden dimension."
            )

        freeze_controlled_backbones(
            self.text_encoder,
            self.vision_encoder,
        )

        if variant == "A":
            self.late_fusion = nn.Linear(
                HIDDEN_DIM * 2,
                HIDDEN_DIM,
            )

            self.fusion_norm = nn.LayerNorm(
                HIDDEN_DIM
            )

        if variant in {
            "B",
            "D",
        }:
            self.cross_t2i = (
                nn.MultiheadAttention(
                    embed_dim=HIDDEN_DIM,
                    num_heads=NUM_HEADS,
                    dropout=ATTN_DROPOUT,
                    batch_first=True,
                )
            )

            self.cross_fusion = nn.Linear(
                HIDDEN_DIM * 2,
                HIDDEN_DIM,
            )

            self.fusion_norm = nn.LayerNorm(
                HIDDEN_DIM
            )

        if variant in {
            "C",
            "C_NO_INCONGRUITY",
            "C_NO_GATE",
        }:
            self.cross_t2i = (
                nn.MultiheadAttention(
                    embed_dim=HIDDEN_DIM,
                    num_heads=NUM_HEADS,
                    dropout=ATTN_DROPOUT,
                    batch_first=True,
                )
            )

            self.cross_i2t = (
                nn.MultiheadAttention(
                    embed_dim=HIDDEN_DIM,
                    num_heads=NUM_HEADS,
                    dropout=ATTN_DROPOUT,
                    batch_first=True,
                )
            )

            self.ca_projection = nn.Linear(
                HIDDEN_DIM * 2,
                HIDDEN_DIM,
            )

            self.ca_norm = nn.LayerNorm(
                HIDDEN_DIM
            )

            if variant in {
                "C",
                "C_NO_GATE",
            }:
                self.text_incongruity_projection = nn.Linear(
                    HIDDEN_DIM,
                    HIDDEN_DIM,
                )

                self.image_incongruity_projection = nn.Linear(
                    HIDDEN_DIM,
                    HIDDEN_DIM,
                )

                self.icam_norm = nn.LayerNorm(
                    HIDDEN_DIM
                )

            if variant == "C":
                self.gate = nn.Linear(
                    HIDDEN_DIM * 2,
                    HIDDEN_DIM,
                )

        if self.uses_topic:
            self.topic_projection = nn.Linear(
                8,
                TOPIC_PROJ_DIM,
            )

            classifier_input = (
                HIDDEN_DIM
                + TOPIC_PROJ_DIM
            )

        else:
            classifier_input = (
                HIDDEN_DIM
            )

        self.classifier = nn.Sequential(
            nn.Linear(
                classifier_input,
                256,
            ),
            nn.GELU(),
            nn.Dropout(
                0.3
            ),
            nn.Linear(
                256,
                2,
            ),
        )

    def semantic_text_mask(
        self,
        input_ids,
        attention_mask,
    ):
        mask = (
            attention_mask.bool().clone()
        )

        for token_id in [
            tokenizer.cls_token_id,
            tokenizer.sep_token_id,
            tokenizer.pad_token_id,
        ]:
            if token_id is not None:
                mask &= input_ids.ne(
                    int(
                        token_id
                    )
                )

        empty = (
            mask.sum(
                dim=1
            )
            .eq(
                0
            )
        )

        if empty.any():
            fallback = (
                attention_mask.bool().clone()
            )

            if tokenizer.cls_token_id is not None:
                fallback &= input_ids.ne(
                    tokenizer.cls_token_id
                )

            mask[
                empty
            ] = fallback[
                empty
            ]

        return mask

    def forward(
        self,
        input_ids,
        attention_mask,
        pixel_values,
        topic_features=None,
        return_diagnostics=False,
    ):
        text_outputs = (
            self.text_encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True,
            )
        )

        vision_outputs = (
            self.vision_encoder(
                pixel_values=pixel_values,
                return_dict=True,
            )
        )

        text_tokens = (
            text_outputs.last_hidden_state
        )

        vision_tokens = (
            vision_outputs.last_hidden_state
        )

        text_cls = (
            text_tokens[
                :,
                0,
                :,
            ]
        )

        image_cls = (
            vision_tokens[
                :,
                0,
                :,
            ]
        )

        image_patches = (
            vision_tokens[
                :,
                1:,
                :,
            ]
        )

        diagnostics = {}

        if self.variant == "A":
            representation = (
                self.fusion_norm(
                    self.late_fusion(
                        torch.cat(
                            [
                                text_cls,
                                image_cls,
                            ],
                            dim=-1,
                        )
                    )
                )
            )

        elif self.variant in {
            "B",
            "D",
        }:
            attended_image, attention_weights = (
                self.cross_t2i(
                    query=text_cls.unsqueeze(
                        1
                    ),
                    key=image_patches,
                    value=image_patches,
                    need_weights=(
                        return_diagnostics
                    ),
                    average_attn_weights=False,
                )
            )

            attended_image = (
                attended_image.squeeze(
                    1
                )
            )

            representation = (
                self.fusion_norm(
                    self.cross_fusion(
                        torch.cat(
                            [
                                text_cls,
                                attended_image,
                            ],
                            dim=-1,
                        )
                    )
                )
            )

            if return_diagnostics:
                diagnostics[
                    "t2i_attention"
                ] = attention_weights

        else:
            z_t2i, t2i_weights = (
                self.cross_t2i(
                    query=text_cls.unsqueeze(
                        1
                    ),
                    key=image_patches,
                    value=image_patches,
                    need_weights=(
                        return_diagnostics
                    ),
                    average_attn_weights=False,
                )
            )

            z_t2i = z_t2i.squeeze(
                1
            )

            z_i2t, i2t_weights = (
                self.cross_i2t(
                    query=image_cls.unsqueeze(
                        1
                    ),
                    key=text_tokens,
                    value=text_tokens,
                    key_padding_mask=(
                        ~attention_mask.bool()
                    ),
                    need_weights=(
                        return_diagnostics
                    ),
                    average_attn_weights=False,
                )
            )

            z_i2t = z_i2t.squeeze(
                1
            )

            z_ca = self.ca_norm(
                self.ca_projection(
                    torch.cat(
                        [
                            z_t2i,
                            z_i2t,
                        ],
                        dim=-1,
                    )
                )
            )

            if self.variant == "C_NO_INCONGRUITY":
                representation = z_ca

                if return_diagnostics:
                    diagnostics.update({
                        "t2i_attention": t2i_weights,
                        "i2t_attention": i2t_weights,
                    })

            else:
                semantic_mask = (
                    self.semantic_text_mask(
                        input_ids,
                        attention_mask,
                    )
                )

                text_projected = (
                    F.normalize(
                        self.text_incongruity_projection(
                            text_tokens
                        ),
                        p=2,
                        dim=-1,
                    )
                )

                image_projected = (
                    F.normalize(
                        self.image_incongruity_projection(
                            image_patches
                        ),
                        p=2,
                        dim=-1,
                    )
                )

                similarity = torch.einsum(
                    "bth,bph->btp",
                    text_projected,
                    image_projected,
                )

                similarity = (
                    similarity.masked_fill(
                        ~semantic_mask.unsqueeze(-1),
                        -1e4,
                    )
                )

                max_similarity = (
                    similarity.max(dim=1).values
                )

                mismatch = 1.0 - max_similarity

                incongruity_weights = torch.softmax(
                    mismatch,
                    dim=-1,
                )

                z_inc = torch.einsum(
                    "bp,bph->bh",
                    incongruity_weights,
                    image_projected,
                )

                if self.variant == "C":
                    gate_values = torch.sigmoid(
                        self.gate(
                            torch.cat(
                                [z_ca, z_inc],
                                dim=-1,
                            )
                        )
                    )

                    representation = self.icam_norm(
                        gate_values * z_ca
                        + (1.0 - gate_values) * z_inc
                    )

                elif self.variant == "C_NO_GATE":
                    representation = self.icam_norm(
                        0.5 * z_ca
                        + 0.5 * z_inc
                    )

                if return_diagnostics:
                    diagnostics.update({
                        "t2i_attention": t2i_weights,
                        "i2t_attention": i2t_weights,
                        "incongruity_weights": incongruity_weights,
                    })

                    if self.variant == "C":
                        diagnostics[
                            "gate_values"
                        ] = gate_values

        if self.uses_topic:
            if topic_features is None:
                raise ValueError(
                    f"Variant {self.variant} membutuhkan topic_features."
                )

            topic_embedding = (
                self.topic_projection(
                    topic_features
                )
            )

            classifier_input = torch.cat(
                [
                    representation,
                    topic_embedding,
                ],
                dim=-1,
            )

        else:
            if topic_features is not None:
                raise ValueError(
                    "Variant D tidak boleh menerima topic_features."
                )

            classifier_input = (
                representation
            )

        logits = self.classifier(
            classifier_input
        )

        if return_diagnostics:
            diagnostics[
                "representation"
            ] = representation

            return (
                logits,
                diagnostics,
            )

        return logits

In [9]:
def trainable_parameter_names(
    model,
):
    return {
        name
        for (
            name,
            parameter,
        )
        in model.named_parameters()
        if parameter.requires_grad
    }

def load_trainable_state(
    model,
    trainable_state,
):
    expected = trainable_parameter_names(
        model
    )

    observed = set(
        trainable_state.keys()
    )

    if expected != observed:
        raise RuntimeError(
            "Trainable checkpoint keys mismatch. "
            f"Missing={sorted(expected-observed)[:10]} "
            f"Unexpected={sorted(observed-expected)[:10]}"
        )

    current = model.state_dict()

    current.update(
        trainable_state
    )

    model.load_state_dict(
        current,
        strict=True,
    )

checkpoint = torch.load(
    REPRESENTATIVE_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

if checkpoint.get(
    "checkpoint_format"
) != "trainable_state_dict_v1":
    raise RuntimeError(
        "Unexpected representative checkpoint format."
    )

if checkpoint[
    "config_hash"
] != actual_protocol_sha:
    raise RuntimeError(
        "Representative checkpoint protocol mismatch."
    )

if (
    checkpoint[
        "variant"
    ]
    != "C"
    or int(
        checkpoint[
            "seed"
        ]
    )
    != EXPECTED_REPRESENTATIVE_SEED
):
    raise RuntimeError(
        "Representative checkpoint identity mismatch."
    )

model = ControlledMultimodalModel(
    variant="C"
)

load_trainable_state(
    model,
    checkpoint[
        "trainable_state"
    ],
)

model = model.to(
    DEVICE
)

model.eval()

assert_freeze_policy(
    model
)

print(
    "Representative C model reconstruction: PASS"
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Representative C model reconstruction: PASS
Trainable parameters: 151415842


## 6. XAI forward helpers

Helper di bawah tidak mengubah parameter model.

- `forward_c_from_tokens`: mereplikasi exact ICAM C fusion setelah encoder;
- `forward_c_from_embeddings`: jalur yang sama tetapi menerima interpolated word embeddings untuk IG;
- ViT rollout dihitung dari actual hidden states yang masuk ke self-attention layers dan exact frozen query/key weights.

In [10]:
def forward_c_from_tokens(
    model,
    text_tokens,
    vision_tokens,
    input_ids,
    attention_mask,
    topic_features,
    need_attention_weights,
):
    text_cls = text_tokens[
        :,
        0,
        :,
    ]

    image_cls = vision_tokens[
        :,
        0,
        :,
    ]

    image_patches = vision_tokens[
        :,
        1:,
        :,
    ]

    z_t2i, t2i_weights = model.cross_t2i(
        query=text_cls.unsqueeze(
            1
        ),
        key=image_patches,
        value=image_patches,
        need_weights=need_attention_weights,
        average_attn_weights=False,
    )

    z_t2i = z_t2i.squeeze(
        1
    )

    z_i2t, i2t_weights = model.cross_i2t(
        query=image_cls.unsqueeze(
            1
        ),
        key=text_tokens,
        value=text_tokens,
        key_padding_mask=(
            ~attention_mask.bool()
        ),
        need_weights=need_attention_weights,
        average_attn_weights=False,
    )

    z_i2t = z_i2t.squeeze(
        1
    )

    z_ca = model.ca_norm(
        model.ca_projection(
            torch.cat(
                [
                    z_t2i,
                    z_i2t,
                ],
                dim=-1,
            )
        )
    )

    semantic_mask = model.semantic_text_mask(
        input_ids,
        attention_mask,
    )

    text_projected = F.normalize(
        model.text_incongruity_projection(
            text_tokens
        ),
        p=2,
        dim=-1,
    )

    image_projected = F.normalize(
        model.image_incongruity_projection(
            image_patches
        ),
        p=2,
        dim=-1,
    )

    similarity = torch.einsum(
        "bth,bph->btp",
        text_projected,
        image_projected,
    )

    similarity = similarity.masked_fill(
        ~semantic_mask.unsqueeze(
            -1
        ),
        -1e4,
    )

    max_similarity = similarity.max(
        dim=1
    ).values

    mismatch = (
        1.0
        - max_similarity
    )

    incongruity_weights = torch.softmax(
        mismatch,
        dim=-1,
    )

    z_inc = torch.einsum(
        "bp,bph->bh",
        incongruity_weights,
        image_projected,
    )

    gate_values = torch.sigmoid(
        model.gate(
            torch.cat(
                [
                    z_ca,
                    z_inc,
                ],
                dim=-1,
            )
        )
    )

    representation = model.icam_norm(
        gate_values
        * z_ca
        + (
            1.0
            - gate_values
        )
        * z_inc
    )

    topic_embedding = model.topic_projection(
        topic_features
    )

    classifier_input = torch.cat(
        [
            representation,
            topic_embedding,
        ],
        dim=-1,
    )

    logits = model.classifier(
        classifier_input
    )

    diagnostics = {
        "semantic_mask": semantic_mask,
        "incongruity_weights": incongruity_weights,
        "gate_values": gate_values,
    }

    if need_attention_weights:
        diagnostics[
            "t2i_attention"
        ] = t2i_weights

        diagnostics[
            "i2t_attention"
        ] = i2t_weights

    return (
        logits,
        diagnostics,
    )

def forward_c_from_embeddings(
    model,
    word_embeddings,
    vision_tokens,
    input_ids,
    attention_mask,
    topic_features,
):
    text_outputs = model.text_encoder(
        inputs_embeds=word_embeddings,
        attention_mask=attention_mask,
        return_dict=True,
    )

    logits, _ = forward_c_from_tokens(
        model=model,
        text_tokens=text_outputs.last_hidden_state,
        vision_tokens=vision_tokens,
        input_ids=input_ids,
        attention_mask=attention_mask,
        topic_features=topic_features,
        need_attention_weights=False,
    )

    return logits

def capture_vision_tokens_and_rollout(
    model,
    pixel_values,
):
    layer_inputs = [
        None
        for _ in model.vision_encoder.encoder.layer
    ]

    handles = []

    for (
        layer_index,
        layer,
    ) in enumerate(
        model.vision_encoder.encoder.layer
    ):
        attention_module = (
            layer.attention.attention
        )

        def make_hook(
            index,
        ):
            def hook(
                module,
                args,
            ):
                layer_inputs[
                    index
                ] = args[
                    0
                ].detach()

            return hook

        handles.append(
            attention_module.register_forward_pre_hook(
                make_hook(
                    layer_index
                )
            )
        )

    try:
        with torch.no_grad():
            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                vision_outputs = model.vision_encoder(
                    pixel_values=pixel_values,
                    return_dict=True,
                )

        vision_tokens = (
            vision_outputs.last_hidden_state
        )

    finally:
        for handle in handles:
            handle.remove()

    if any(
        value is None
        for value in layer_inputs
    ):
        raise RuntimeError(
            "Failed to capture all ViT self-attention inputs."
        )

    num_heads = int(
        model.vision_encoder.config.num_attention_heads
    )

    hidden_size = int(
        model.vision_encoder.config.hidden_size
    )

    head_dim = (
        hidden_size
        // num_heads
    )

    joint_attention = None

    for (
        layer,
        hidden_states,
    ) in zip(
        model.vision_encoder.encoder.layer,
        layer_inputs,
    ):
        attention_module = (
            layer.attention.attention
        )

        batch_size, sequence_length, _ = (
            hidden_states.shape
        )

        with torch.no_grad():
            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                queries = attention_module.query(
                    hidden_states
                )

                keys = attention_module.key(
                    hidden_states
                )

        queries = (
            queries
            .view(
                batch_size,
                sequence_length,
                num_heads,
                head_dim,
            )
            .permute(
                0,
                2,
                1,
                3,
            )
            .float()
        )

        keys = (
            keys
            .view(
                batch_size,
                sequence_length,
                num_heads,
                head_dim,
            )
            .permute(
                0,
                2,
                1,
                3,
            )
            .float()
        )

        attention_scores = torch.matmul(
            queries,
            keys.transpose(
                -2,
                -1,
            ),
        ) / math.sqrt(
            float(
                head_dim
            )
        )

        attention_probs = torch.softmax(
            attention_scores,
            dim=-1,
        )

        mean_attention = attention_probs.mean(
            dim=1
        )

        identity = torch.eye(
            sequence_length,
            device=mean_attention.device,
            dtype=mean_attention.dtype,
        ).unsqueeze(
            0
        )

        augmented = (
            mean_attention
            + identity
        )

        augmented = (
            augmented
            / augmented.sum(
                dim=-1,
                keepdim=True,
            ).clamp_min(
                1e-12
            )
        )

        if joint_attention is None:
            joint_attention = augmented

        else:
            joint_attention = torch.bmm(
                augmented,
                joint_attention,
            )

    rollout = joint_attention[
        :,
        0,
        1:,
    ]

    rollout = (
        rollout
        / rollout.sum(
            dim=-1,
            keepdim=True,
        ).clamp_min(
            1e-12
        )
    )

    return (
        vision_tokens.detach(),
        rollout.detach(),
    )

print(
    "XAI forward helpers: READY"
)

XAI forward helpers: READY


## 7. Integrated Gradients — 50 steps

Baseline untuk word-embedding path adalah **PAD-token word embedding** pada posisi yang sama, sementara attention mask, positional embedding, image input, topic vector, dan seluruh parameter model tetap fixed.

Scientific IG target tetap frozen: **argmax predicted-class logit**, bukan thresholded class.

Approximation menggunakan 50 equally spaced right-Riemann points. `IG_STEP_CHUNK` hanya mengatur penggunaan memori; seluruh 50 points tetap dihitung.

In [11]:
def integrated_gradients_text_batch(
    model,
    input_ids,
    attention_mask,
    topic_features,
    vision_tokens,
    target_classes,
    steps=50,
    step_chunk=10,
):
    batch_size = int(
        input_ids.shape[
            0
        ]
    )

    embedding_layer = (
        model.text_encoder.get_input_embeddings()
    )

    with torch.no_grad():
        input_word_embeddings = embedding_layer(
            input_ids
        ).detach()

        if (
            tokenizer.pad_token_id
            is not None
        ):
            baseline_ids = torch.full_like(
                input_ids,
                int(
                    tokenizer.pad_token_id
                ),
            )

            baseline_word_embeddings = embedding_layer(
                baseline_ids
            ).detach()

        else:
            baseline_word_embeddings = torch.zeros_like(
                input_word_embeddings
            )

    delta = (
        input_word_embeddings
        - baseline_word_embeddings
    )

    gradient_sum = torch.zeros_like(
        input_word_embeddings,
        dtype=torch.float32,
    )

    alpha_values = torch.arange(
        1,
        steps + 1,
        device=DEVICE,
        dtype=torch.float32,
    ) / float(
        steps
    )

    for start in range(
        0,
        steps,
        step_chunk,
    ):
        alphas = alpha_values[
            start:
            start + step_chunk
        ]

        chunk_size = int(
            len(
                alphas
            )
        )

        interpolated = (
            baseline_word_embeddings.unsqueeze(
                0
            )
            + alphas.view(
                chunk_size,
                1,
                1,
                1,
            )
            * delta.unsqueeze(
                0
            )
        )

        # [K, B, L, H] -> [K*B, L, H]
        interpolated = (
            interpolated
            .reshape(
                chunk_size
                * batch_size,
                input_word_embeddings.shape[
                    1
                ],
                input_word_embeddings.shape[
                    2
                ],
            )
            .detach()
            .requires_grad_(
                True
            )
        )

        repeated_input_ids = (
            input_ids
            .unsqueeze(
                0
            )
            .expand(
                chunk_size,
                -1,
                -1,
            )
            .reshape(
                chunk_size
                * batch_size,
                -1,
            )
        )

        repeated_attention_mask = (
            attention_mask
            .unsqueeze(
                0
            )
            .expand(
                chunk_size,
                -1,
                -1,
            )
            .reshape(
                chunk_size
                * batch_size,
                -1,
            )
        )

        repeated_topic = (
            topic_features
            .unsqueeze(
                0
            )
            .expand(
                chunk_size,
                -1,
                -1,
            )
            .reshape(
                chunk_size
                * batch_size,
                -1,
            )
        )

        repeated_vision = (
            vision_tokens
            .unsqueeze(
                0
            )
            .expand(
                chunk_size,
                -1,
                -1,
                -1,
            )
            .reshape(
                chunk_size
                * batch_size,
                vision_tokens.shape[
                    1
                ],
                vision_tokens.shape[
                    2
                ],
            )
        )

        repeated_targets = (
            target_classes
            .unsqueeze(
                0
            )
            .expand(
                chunk_size,
                -1,
            )
            .reshape(
                -1
            )
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            logits = forward_c_from_embeddings(
                model=model,
                word_embeddings=interpolated,
                vision_tokens=repeated_vision,
                input_ids=repeated_input_ids,
                attention_mask=repeated_attention_mask,
                topic_features=repeated_topic,
            )

            target_logits = logits.gather(
                1,
                repeated_targets.unsqueeze(
                    1
                ),
            ).squeeze(
                1
            )

        gradients = torch.autograd.grad(
            outputs=target_logits.sum(),
            inputs=interpolated,
            retain_graph=False,
            create_graph=False,
            allow_unused=False,
        )[
            0
        ]

        gradients = (
            gradients
            .float()
            .reshape(
                chunk_size,
                batch_size,
                input_word_embeddings.shape[
                    1
                ],
                input_word_embeddings.shape[
                    2
                ],
            )
        )

        gradient_sum += gradients.sum(
            dim=0
        )

        del (
            interpolated,
            logits,
            target_logits,
            gradients,
            repeated_input_ids,
            repeated_attention_mask,
            repeated_topic,
            repeated_vision,
            repeated_targets,
        )

    average_gradient = (
        gradient_sum
        / float(
            steps
        )
    )

    integrated = (
        delta.float()
        * average_gradient
    )

    # Common token-level scalarization:
    # signed sum over embedding dimensions, then absolute value for ranking.
    signed_token_ig = integrated.sum(
        dim=-1
    )

    absolute_token_ig = signed_token_ig.abs()

    l1_token_ig = integrated.abs().sum(
        dim=-1
    )

    return {
        "signed": signed_token_ig.detach(),
        "absolute": absolute_token_ig.detach(),
        "l1": l1_token_ig.detach(),
    }

print(
    "Integrated Gradients helper: READY"
)

Integrated Gradients helper: READY


## 8. End-to-end probe against frozen official-test prediction

Sebelum expensive IG run, reconstructed model harus mereproduksi probability C seed 999 dari final official-test artifact.

In [12]:
official_predictions = pd.read_csv(
    OFFICIAL_PREDICTION_PATH
)

required_prediction_columns = {
    "sample_uid",
    "label",
    "prob_hateful",
}

if not required_prediction_columns.issubset(
    set(
        official_predictions.columns
    )
):
    raise RuntimeError(
        "Frozen official prediction columns invalid."
    )

official_predictions = (
    official_predictions[
        [
            "sample_uid",
            "label",
            "prob_hateful",
        ]
    ]
    .sort_values(
        "sample_uid"
    )
    .reset_index(
        drop=True
    )
)

probe_loader = DataLoader(
    torch.utils.data.Subset(
        xai_dataset,
        list(
            range(
                min(
                    XAI_BATCH_SIZE,
                    len(
                        xai_dataset
                    ),
                )
            )
        ),
    ),
    batch_size=XAI_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_xai,
)

probe = next(
    iter(
        probe_loader
    )
)

probe_input_ids = probe[
    "input_ids"
].to(
    DEVICE
)

probe_attention_mask = probe[
    "attention_mask"
].to(
    DEVICE
)

probe_pixel_values = probe[
    "pixel_values"
].to(
    DEVICE
)

probe_topic = probe[
    "topic_features"
].to(
    DEVICE
)

vision_tokens, rollout = (
    capture_vision_tokens_and_rollout(
        model,
        probe_pixel_values,
    )
)

with torch.no_grad():
    with torch.amp.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):
        text_outputs = model.text_encoder(
            input_ids=probe_input_ids,
            attention_mask=probe_attention_mask,
            return_dict=True,
        )

        manual_logits, manual_diagnostics = (
            forward_c_from_tokens(
                model=model,
                text_tokens=text_outputs.last_hidden_state,
                vision_tokens=vision_tokens,
                input_ids=probe_input_ids,
                attention_mask=probe_attention_mask,
                topic_features=probe_topic,
                need_attention_weights=True,
            )
        )

        native_logits, native_diagnostics = model(
            input_ids=probe_input_ids,
            attention_mask=probe_attention_mask,
            pixel_values=probe_pixel_values,
            topic_features=probe_topic,
            return_diagnostics=True,
        )

manual_probs = torch.softmax(
    manual_logits.float(),
    dim=-1,
)[:, 1]

native_probs = torch.softmax(
    native_logits.float(),
    dim=-1,
)[:, 1]

manual_native_diff = float(
    (
        manual_probs
        - native_probs
    )
    .abs()
    .max()
    .item()
)

frozen_probe = (
    official_predictions
    .set_index(
        "sample_uid"
    )
    .loc[
        probe[
            "sample_uid"
        ]
    ][
        "prob_hateful"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

manual_frozen_diff = float(
    np.max(
        np.abs(
            manual_probs.detach().cpu().numpy()
            - frozen_probe
        )
    )
)

if (
    manual_native_diff
    > 5e-4
):
    raise RuntimeError(
        f"Manual-vs-native ICAM probability mismatch: {manual_native_diff}"
    )

if (
    manual_frozen_diff
    > 5e-3
):
    raise RuntimeError(
        f"Reconstructed-vs-frozen official prediction mismatch: {manual_frozen_diff}"
    )

if rollout.shape[
    1
] != 196:
    raise RuntimeError(
        f"Expected 196 rollout patches, got {rollout.shape}"
    )

if manual_diagnostics[
    "t2i_attention"
].shape[
    -1
] != 196:
    raise RuntimeError(
        "Cross-attention patch dimension mismatch."
    )

if manual_diagnostics[
    "incongruity_weights"
].shape[
    -1
] != 196:
    raise RuntimeError(
        "Incongruity patch dimension mismatch."
    )

probe_report = {
    "manual_native_probability_max_abs_diff": manual_native_diff,
    "manual_frozen_probability_max_abs_diff": manual_frozen_diff,
    "rollout_patch_count": int(
        rollout.shape[
            1
        ]
    ),
    "cross_attention_patch_count": int(
        manual_diagnostics[
            "t2i_attention"
        ].shape[
            -1
        ]
    ),
    "incongruity_patch_count": int(
        manual_diagnostics[
            "incongruity_weights"
        ].shape[
            -1
        ]
    ),
    "status": "PASS",
}

(
    OUTPUT_ROOT
    / "xai_reconstruction_probe.json"
).write_text(
    json.dumps(
        probe_report,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        probe_report,
        indent=2,
    )
)

del (
    probe_loader,
    probe,
    probe_input_ids,
    probe_attention_mask,
    probe_pixel_values,
    probe_topic,
    vision_tokens,
    rollout,
    text_outputs,
    manual_logits,
    native_logits,
    manual_diagnostics,
    native_diagnostics,
    manual_probs,
    native_probs,
)

gc.collect()
torch.cuda.empty_cache()

{
  "manual_native_probability_max_abs_diff": 0.0,
  "manual_frozen_probability_max_abs_diff": 0.00013218820095062256,
  "rollout_patch_count": 196,
  "cross_attention_patch_count": 196,
  "incongruity_patch_count": 196,
  "status": "PASS"
}


## 9. Process one XAI batch

In [13]:
def process_xai_batch(
    batch,
):
    input_ids = batch[
        "input_ids"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    attention_mask = batch[
        "attention_mask"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    pixel_values = batch[
        "pixel_values"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    topic_features = batch[
        "topic_features"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    labels = batch[
        "labels"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    vision_tokens, rollout = (
        capture_vision_tokens_and_rollout(
            model,
            pixel_values,
        )
    )

    with torch.no_grad():
        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            text_outputs = model.text_encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True,
            )

            logits, diagnostics = (
                forward_c_from_tokens(
                    model=model,
                    text_tokens=text_outputs.last_hidden_state,
                    vision_tokens=vision_tokens,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    topic_features=topic_features,
                    need_attention_weights=True,
                )
            )

    probabilities = torch.softmax(
        logits.float(),
        dim=-1,
    )

    hateful_probabilities = probabilities[
        :,
        1
    ]

    predicted_classes = logits.argmax(
        dim=-1
    )

    frozen_threshold_predictions = (
        hateful_probabilities
        >= REPRESENTATIVE_THRESHOLD
    ).long()

    ig = integrated_gradients_text_batch(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        topic_features=topic_features,
        vision_tokens=vision_tokens.detach(),
        target_classes=predicted_classes.detach(),
        steps=IG_STEPS,
        step_chunk=IG_STEP_CHUNK,
    )

    semantic_mask = diagnostics[
        "semantic_mask"
    ]

    # Frozen protocol: IG is interpreted/ranked only on semantic tokens.
    ig_signed = ig[
        "signed"
    ].masked_fill(
        ~semantic_mask,
        0.0,
    )

    ig_absolute = ig[
        "absolute"
    ].masked_fill(
        ~semantic_mask,
        0.0,
    )

    ig_l1 = ig[
        "l1"
    ].masked_fill(
        ~semantic_mask,
        0.0,
    )

    t2i_per_head = diagnostics[
        "t2i_attention"
    ].squeeze(
        2
    ).float()

    t2i_mean = t2i_per_head.mean(
        dim=1
    )

    i2t_mean = diagnostics[
        "i2t_attention"
    ].squeeze(
        2
    ).float().mean(
        dim=1
    )

    incongruity = diagnostics[
        "incongruity_weights"
    ].float()

    gate_values = diagnostics[
        "gate_values"
    ].float()

    return {
        "sample_uid": list(
            batch[
                "sample_uid"
            ]
        ),
        "labels": labels.detach().cpu().numpy().astype(
            np.int64
        ),
        "input_ids": input_ids.detach().cpu().numpy().astype(
            np.int32
        ),
        "attention_mask": attention_mask.detach().cpu().numpy().astype(
            np.uint8
        ),
        "semantic_mask": semantic_mask.detach().cpu().numpy().astype(
            np.uint8
        ),
        "hateful_probability": hateful_probabilities.detach().cpu().numpy().astype(
            np.float32
        ),
        "predicted_class_argmax": predicted_classes.detach().cpu().numpy().astype(
            np.int64
        ),
        "prediction_frozen_threshold": frozen_threshold_predictions.detach().cpu().numpy().astype(
            np.int64
        ),
        "ig_signed": ig_signed.detach().cpu().numpy().astype(
            np.float32
        ),
        "ig_absolute": ig_absolute.detach().cpu().numpy().astype(
            np.float32
        ),
        "ig_l1": ig_l1.detach().cpu().numpy().astype(
            np.float32
        ),
        "cross_attention_per_head": t2i_per_head.detach().cpu().numpy().astype(
            np.float32
        ),
        "cross_attention_mean": t2i_mean.detach().cpu().numpy().astype(
            np.float32
        ),
        "i2t_attention_mean": i2t_mean.detach().cpu().numpy().astype(
            np.float32
        ),
        "attention_rollout": rollout.detach().cpu().numpy().astype(
            np.float32
        ),
        "incongruity": incongruity.detach().cpu().numpy().astype(
            np.float32
        ),
        "gate_mean": gate_values.mean(
            dim=-1
        ).detach().cpu().numpy().astype(
            np.float32
        ),
        "gate_std": gate_values.std(
            dim=-1,
            unbiased=False,
        ).detach().cpu().numpy().astype(
            np.float32
        ),
    }

print(
    "Batch XAI function: READY"
)

Batch XAI function: READY


## 10. Sharded all-test XAI execution

Setiap shard berisi maksimal 25 samples. Jika cell dijalankan ulang dalam session yang sama, shard yang sudah valid akan dilewati.

In [14]:
def save_npz_atomic(
    path,
    arrays,
):
    path = Path(
        path
    )

    tmp = path.with_suffix(
        ".tmp.npz"
    )

    np.savez_compressed(
        tmp,
        **arrays,
    )

    os.replace(
        tmp,
        path,
    )

def concatenate_batch_outputs(
    outputs,
):
    combined = {}

    combined[
        "sample_uid"
    ] = np.asarray(
        [
            uid
            for output in outputs
            for uid in output[
                "sample_uid"
            ]
        ],
        dtype=str,
    )

    array_keys = [
        "labels",
        "input_ids",
        "attention_mask",
        "semantic_mask",
        "hateful_probability",
        "predicted_class_argmax",
        "prediction_frozen_threshold",
        "ig_signed",
        "ig_absolute",
        "ig_l1",
        "cross_attention_per_head",
        "cross_attention_mean",
        "i2t_attention_mean",
        "attention_rollout",
        "incongruity",
        "gate_mean",
        "gate_std",
    ]

    for key in array_keys:
        combined[
            key
        ] = np.concatenate(
            [
                output[
                    key
                ]
                for output in outputs
            ],
            axis=0,
        )

    return combined

def shard_is_valid(
    shard_path,
    expected_uids,
):
    if not shard_path.exists():
        return False

    try:
        with np.load(
            shard_path,
            allow_pickle=False,
        ) as data:
            observed_uids = (
                data[
                    "sample_uid"
                ]
                .astype(
                    str
                )
                .tolist()
            )

            if observed_uids != list(
                expected_uids
            ):
                return False

            required = {
                "ig_absolute",
                "cross_attention_mean",
                "attention_rollout",
                "incongruity",
            }

            if not required.issubset(
                set(
                    data.files
                )
            ):
                return False

        return True

    except Exception:
        return False

total_samples = len(
    xai_dataset
)

num_shards = math.ceil(
    total_samples
    / SHARD_SIZE
)

run_start = time.perf_counter()

for shard_index in range(
    num_shards
):
    start_index = (
        shard_index
        * SHARD_SIZE
    )

    end_index = min(
        start_index
        + SHARD_SIZE,
        total_samples,
    )

    expected_uids = (
        test_df.iloc[
            start_index:
            end_index
        ][
            "sample_uid"
        ]
        .astype(
            str
        )
        .tolist()
    )

    shard_path = (
        SHARD_ROOT
        / f"xai_shard_{shard_index:03d}.npz"
    )

    if shard_is_valid(
        shard_path,
        expected_uids,
    ):
        print(
            f"[SKIP] shard {shard_index:03d} "
            f"samples {start_index}:{end_index}"
        )

        continue

    print(
        "\n"
        + "="
        * 72
    )

    print(
        f"XAI shard {shard_index + 1}/{num_shards} "
        f"samples {start_index}:{end_index}"
    )

    print(
        "="
        * 72
    )

    subset = torch.utils.data.Subset(
        xai_dataset,
        list(
            range(
                start_index,
                end_index,
            )
        ),
    )

    loader = DataLoader(
        subset,
        batch_size=XAI_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_xai,
    )

    batch_outputs = []

    for batch_index, batch in enumerate(
        loader
    ):
        batch_start = time.perf_counter()

        output = process_xai_batch(
            batch
        )

        batch_outputs.append(
            output
        )

        elapsed = (
            time.perf_counter()
            - batch_start
        )

        print(
            f"  batch {batch_index + 1}/{len(loader)} "
            f"n={len(output['sample_uid'])} "
            f"{elapsed:.1f}s"
        )

        gc.collect()
        torch.cuda.empty_cache()

    shard_arrays = concatenate_batch_outputs(
        batch_outputs
    )

    if (
        shard_arrays[
            "sample_uid"
        ].astype(
            str
        ).tolist()
        != expected_uids
    ):
        raise RuntimeError(
            f"Shard UID alignment failure: {shard_index}"
        )

    save_npz_atomic(
        shard_path,
        shard_arrays,
    )

    del (
        subset,
        loader,
        batch_outputs,
        shard_arrays,
    )

    gc.collect()
    torch.cuda.empty_cache()

total_runtime_seconds = (
    time.perf_counter()
    - run_start
)

print(
    "\nAll XAI shards complete."
)

print(
    "Runtime seconds:",
    total_runtime_seconds,
)


XAI shard 1/41 samples 0:25
  batch 1/13 n=2 0.7s
  batch 2/13 n=2 0.4s
  batch 3/13 n=2 0.4s
  batch 4/13 n=2 0.4s
  batch 5/13 n=2 0.4s
  batch 6/13 n=2 0.4s
  batch 7/13 n=2 0.4s
  batch 8/13 n=2 0.4s
  batch 9/13 n=2 0.4s
  batch 10/13 n=2 0.4s
  batch 11/13 n=2 0.4s
  batch 12/13 n=2 0.4s
  batch 13/13 n=1 0.3s

XAI shard 2/41 samples 25:50
  batch 1/13 n=2 0.4s
  batch 2/13 n=2 0.4s
  batch 3/13 n=2 0.4s
  batch 4/13 n=2 0.4s
  batch 5/13 n=2 0.4s
  batch 6/13 n=2 0.4s
  batch 7/13 n=2 0.4s
  batch 8/13 n=2 0.4s
  batch 9/13 n=2 0.4s
  batch 10/13 n=2 0.4s
  batch 11/13 n=2 0.4s
  batch 12/13 n=2 0.4s
  batch 13/13 n=1 0.2s

XAI shard 3/41 samples 50:75
  batch 1/13 n=2 0.4s
  batch 2/13 n=2 0.4s
  batch 3/13 n=2 0.4s
  batch 4/13 n=2 0.4s
  batch 5/13 n=2 0.4s
  batch 6/13 n=2 0.4s
  batch 7/13 n=2 0.4s
  batch 8/13 n=2 0.4s
  batch 9/13 n=2 0.4s
  batch 10/13 n=2 0.4s
  batch 11/13 n=2 0.4s
  batch 12/13 n=2 0.4s
  batch 13/13 n=1 0.3s

XAI shard 4/41 samples 75:100
  batch 1/

## 11. Aggregate shards menjadi final XAI artifact

In [15]:
all_arrays = []

for shard_index in range(
    num_shards
):
    shard_path = (
        SHARD_ROOT
        / f"xai_shard_{shard_index:03d}.npz"
    )

    if not shard_path.exists():
        raise RuntimeError(
            f"Missing XAI shard: {shard_path}"
        )

    with np.load(
        shard_path,
        allow_pickle=False,
    ) as data:
        all_arrays.append({
            key: data[
                key
            ].copy()
            for key in data.files
        })

final_arrays = {}

final_arrays[
    "sample_uid"
] = np.concatenate(
    [
        arrays[
            "sample_uid"
        ]
        for arrays in all_arrays
    ],
    axis=0,
)

for key in [
    "labels",
    "input_ids",
    "attention_mask",
    "semantic_mask",
    "hateful_probability",
    "predicted_class_argmax",
    "prediction_frozen_threshold",
    "ig_signed",
    "ig_absolute",
    "ig_l1",
    "cross_attention_per_head",
    "cross_attention_mean",
    "i2t_attention_mean",
    "attention_rollout",
    "incongruity",
    "gate_mean",
    "gate_std",
]:
    final_arrays[
        key
    ] = np.concatenate(
        [
            arrays[
                key
            ]
            for arrays in all_arrays
        ],
        axis=0,
    )

if len(
    final_arrays[
        "sample_uid"
    ]
) != 1005:
    raise RuntimeError(
        "Final XAI sample count mismatch."
    )

expected_uids = (
    test_df[
        "sample_uid"
    ]
    .astype(
        str
    )
    .to_numpy()
)

if not np.array_equal(
    final_arrays[
        "sample_uid"
    ].astype(
        str
    ),
    expected_uids,
):
    raise RuntimeError(
        "Final XAI sample order mismatch."
    )

FINAL_NPZ = (
    OUTPUT_ROOT
    / "representative_C_seed999_xai_all_test.npz"
)

save_npz_atomic(
    FINAL_NPZ,
    final_arrays,
)

print(
    "Final XAI NPZ:",
    FINAL_NPZ
)

print(
    "Size MB:",
    round(
        FINAL_NPZ.stat().st_size
        / (
            1024
            ** 2
        ),
        2,
    )
)

Final XAI NPZ: /kaggle/working/indomeme_xai_revision_v3/xai_representative_C_seed999/representative_C_seed999_xai_all_test.npz
Size MB: 7.44


## 12. Build human-readable sample summary dan top-feature table

Patch indices menggunakan raster order `0..195` pada grid `14×14`.

In [16]:
def top_k_indices(
    values,
    k=10,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    k = min(
        int(
            k
        ),
        len(
            values
        ),
    )

    return np.argsort(
        -values,
        kind="stable",
    )[
        :k
    ]

summary_rows = []

for index in range(
    len(
        final_arrays[
            "sample_uid"
        ]
    )
):
    uid = str(
        final_arrays[
            "sample_uid"
        ][
            index
        ]
    )

    input_ids_row = final_arrays[
        "input_ids"
    ][
        index
    ]

    semantic_mask_row = final_arrays[
        "semantic_mask"
    ][
        index
    ].astype(
        bool
    )

    semantic_positions = np.flatnonzero(
        semantic_mask_row
    )

    ig_values = final_arrays[
        "ig_absolute"
    ][
        index
    ]

    ranked_semantic_positions = semantic_positions[
        np.argsort(
            -ig_values[
                semantic_positions
            ],
            kind="stable",
        )
    ][
        :10
    ]

    top_tokens = []

    for position in ranked_semantic_positions:
        token_id = int(
            input_ids_row[
                position
            ]
        )

        top_tokens.append({
            "position": int(
                position
            ),
            "token_id": token_id,
            "token": tokenizer.convert_ids_to_tokens(
                token_id
            ),
            "ig_signed": float(
                final_arrays[
                    "ig_signed"
                ][
                    index,
                    position
                ]
            ),
            "ig_absolute": float(
                final_arrays[
                    "ig_absolute"
                ][
                    index,
                    position
                ]
            ),
        })

    top_cross = top_k_indices(
        final_arrays[
            "cross_attention_mean"
        ][
            index
        ],
        10,
    )

    top_rollout = top_k_indices(
        final_arrays[
            "attention_rollout"
        ][
            index
        ],
        10,
    )

    top_incongruity = top_k_indices(
        final_arrays[
            "incongruity"
        ][
            index
        ],
        10,
    )

    label = int(
        final_arrays[
            "labels"
        ][
            index
        ]
    )

    argmax_prediction = int(
        final_arrays[
            "predicted_class_argmax"
        ][
            index
        ]
    )

    frozen_prediction = int(
        final_arrays[
            "prediction_frozen_threshold"
        ][
            index
        ]
    )

    summary_rows.append({
        "sample_uid": uid,
        "label": label,
        "hateful_probability": float(
            final_arrays[
                "hateful_probability"
            ][
                index
            ]
        ),
        "predicted_class_argmax": argmax_prediction,
        "prediction_frozen_threshold": frozen_prediction,
        "correct_argmax": bool(
            argmax_prediction
            == label
        ),
        "correct_frozen_threshold": bool(
            frozen_prediction
            == label
        ),
        "gate_mean": float(
            final_arrays[
                "gate_mean"
            ][
                index
            ]
        ),
        "gate_std": float(
            final_arrays[
                "gate_std"
            ][
                index
            ]
        ),
        "top_ig_tokens_json": json.dumps(
            top_tokens,
            ensure_ascii=False,
        ),
        "top_cross_attention_patches_json": json.dumps(
            [
                int(
                    value
                )
                for value in top_cross
            ]
        ),
        "top_attention_rollout_patches_json": json.dumps(
            [
                int(
                    value
                )
                for value in top_rollout
            ]
        ),
        "top_incongruity_patches_json": json.dumps(
            [
                int(
                    value
                )
                for value in top_incongruity
            ]
        ),
    })

summary_df = pd.DataFrame(
    summary_rows
)

SUMMARY_CSV = (
    OUTPUT_ROOT
    / "representative_C_seed999_xai_sample_summary.csv"
)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
)

print(
    summary_df.head(
        3
    ).to_string(
        index=False
    )
)

                     sample_uid  label  hateful_probability  predicted_class_argmax  prediction_frozen_threshold  correct_argmax  correct_frozen_threshold  gate_mean  gate_std                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

## 13. XAI integrity audit

Audit tidak menilai apakah explanation “bagus”. Ia hanya memverifikasi correctness, completeness, dan numerical validity.

In [17]:
official_lookup = (
    official_predictions
    .set_index(
        "sample_uid"
    )
    .loc[
        final_arrays[
            "sample_uid"
        ].astype(
            str
        )
    ]
)

probability_difference = np.abs(
    final_arrays[
        "hateful_probability"
    ].astype(
        np.float64
    )
    - official_lookup[
        "prob_hateful"
    ].to_numpy(
        dtype=np.float64
    )
)

semantic_mask = final_arrays[
    "semantic_mask"
].astype(
    bool
)

nonsemantic_ig_abs_max = float(
    np.max(
        np.abs(
            final_arrays[
                "ig_absolute"
            ][
                ~semantic_mask
            ]
        )
    )
)

cross_sums = final_arrays[
    "cross_attention_mean"
].sum(
    axis=1
)

rollout_sums = final_arrays[
    "attention_rollout"
].sum(
    axis=1
)

incongruity_sums = final_arrays[
    "incongruity"
].sum(
    axis=1
)

numeric_arrays = [
    "hateful_probability",
    "ig_signed",
    "ig_absolute",
    "ig_l1",
    "cross_attention_per_head",
    "cross_attention_mean",
    "i2t_attention_mean",
    "attention_rollout",
    "incongruity",
    "gate_mean",
    "gate_std",
]

nan_counts = {
    key: int(
        np.isnan(
            final_arrays[
                key
            ]
        ).sum()
    )
    for key in numeric_arrays
}

inf_counts = {
    key: int(
        np.isinf(
            final_arrays[
                key
            ]
        ).sum()
    )
    for key in numeric_arrays
}

audit = {
    "status": "PASS",
    "variant": "C",
    "representative_seed": 999,
    "checkpoint_sha256": checkpoint_sha,
    "sample_count": int(
        len(
            final_arrays[
                "sample_uid"
            ]
        )
    ),
    "ig": {
        "steps": IG_STEPS,
        "target": "predicted-class logit",
        "baseline": "PAD-token word embedding at all positions",
        "integration_rule": "50 equally spaced right-Riemann points",
        "semantic_token_only_reporting_and_ranking": True,
        "nonsemantic_ig_absolute_max_after_mask": nonsemantic_ig_abs_max,
    },
    "cross_attention": {
        "direction": "text CLS -> 196 image patches",
        "per_head_saved": True,
        "head_count": int(
            final_arrays[
                "cross_attention_per_head"
            ].shape[
                1
            ]
        ),
        "patch_count": int(
            final_arrays[
                "cross_attention_mean"
            ].shape[
                1
            ]
        ),
        "mean_sum_min": float(
            cross_sums.min()
        ),
        "mean_sum_max": float(
            cross_sums.max()
        ),
    },
    "attention_rollout": {
        "residual_identity": True,
        "patch_count": int(
            final_arrays[
                "attention_rollout"
            ].shape[
                1
            ]
        ),
        "sum_min": float(
            rollout_sums.min()
        ),
        "sum_max": float(
            rollout_sums.max()
        ),
    },
    "incongruity": {
        "source": "internal ICAM a_j",
        "patch_count": int(
            final_arrays[
                "incongruity"
            ].shape[
                1
            ]
        ),
        "sum_min": float(
            incongruity_sums.min()
        ),
        "sum_max": float(
            incongruity_sums.max()
        ),
    },
    "official_prediction_reproduction": {
        "max_abs_probability_difference": float(
            probability_difference.max()
        ),
        "mean_abs_probability_difference": float(
            probability_difference.mean()
        ),
    },
    "nan_counts": nan_counts,
    "inf_counts": inf_counts,
}

if audit[
    "sample_count"
] != 1005:
    audit[
        "status"
    ] = "FAIL"

if any(
    value != 0
    for value in nan_counts.values()
):
    audit[
        "status"
    ] = "FAIL"

if any(
    value != 0
    for value in inf_counts.values()
):
    audit[
        "status"
    ] = "FAIL"

if (
    audit[
        "official_prediction_reproduction"
    ][
        "max_abs_probability_difference"
    ]
    > 5e-3
):
    audit[
        "status"
    ] = "FAIL"

if not (
    0.999
    <= float(
        cross_sums.min()
    )
    <= 1.001
    and 0.999
    <= float(
        cross_sums.max()
    )
    <= 1.001
):
    audit[
        "status"
    ] = "FAIL"

if not (
    0.999
    <= float(
        rollout_sums.min()
    )
    <= 1.001
    and 0.999
    <= float(
        rollout_sums.max()
    )
    <= 1.001
):
    audit[
        "status"
    ] = "FAIL"

if not (
    0.999
    <= float(
        incongruity_sums.min()
    )
    <= 1.001
    and 0.999
    <= float(
        incongruity_sums.max()
    )
    <= 1.001
):
    audit[
        "status"
    ] = "FAIL"

AUDIT_PATH = (
    OUTPUT_ROOT
    / "XAI_METHOD_AUDIT.json"
)

AUDIT_PATH.write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    )
)

if audit[
    "status"
] != "PASS":
    raise RuntimeError(
        "XAI method audit FAIL."
    )

{
  "status": "PASS",
  "variant": "C",
  "representative_seed": 999,
  "checkpoint_sha256": "fc76e1b4bf2288e99802356d63a8ad5345cef54732767c25c9e51178d21520c4",
  "sample_count": 1005,
  "ig": {
    "steps": 50,
    "target": "predicted-class logit",
    "baseline": "PAD-token word embedding at all positions",
    "integration_rule": "50 equally spaced right-Riemann points",
    "semantic_token_only_reporting_and_ranking": true,
    "nonsemantic_ig_absolute_max_after_mask": 0.0
  },
  "cross_attention": {
    "direction": "text CLS -> 196 image patches",
    "per_head_saved": true,
    "head_count": 8,
    "patch_count": 196,
    "mean_sum_min": 0.9999998807907104,
    "mean_sum_max": 1.0000001192092896
  },
  "attention_rollout": {
    "residual_identity": true,
    "patch_count": 196,
    "sum_min": 0.9999998211860657,
    "sum_max": 1.0000001192092896
  },
  "incongruity": {
    "source": "internal ICAM a_j",
    "patch_count": 196,
    "sum_min": 0.9999998807907104,
    "sum_max": 

## 14. Artifact manifest + final status

In [18]:
artifact_paths = {
    "xai_npz": FINAL_NPZ,
    "sample_summary_csv": SUMMARY_CSV,
    "method_audit_json": AUDIT_PATH,
    "reconstruction_probe_json": (
        OUTPUT_ROOT
        / "xai_reconstruction_probe.json"
    ),
}

artifact_manifest = {
    "protocol_sha256": actual_protocol_sha,
    "freeze_record_sha256": actual_freeze_sha,
    "final_official_test_record_sha256": actual_final_test_sha,
    "representative_checkpoint_sha256": checkpoint_sha,
    "artifacts": {
        name: {
            "path": str(
                path
            ),
            "sha256": sha256_file(
                path
            ),
            "size_bytes": int(
                path.stat().st_size
            ),
        }
        for (
            name,
            path,
        )
        in artifact_paths.items()
    },
}

ARTIFACT_MANIFEST_PATH = (
    OUTPUT_ROOT
    / "xai_artifact_manifest.json"
)

ARTIFACT_MANIFEST_PATH.write_text(
    json.dumps(
        artifact_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

status = {
    "status": "PASS",
    "phase": "XAI_REPRESENTATIVE_C",
    "variant": "C",
    "representative_seed": 999,
    "representative_selection_source": "frozen validation-only rule",
    "representative_threshold": REPRESENTATIVE_THRESHOLD,
    "checkpoint_sha256": checkpoint_sha,
    "protocol_sha256": actual_protocol_sha,
    "freeze_record_sha256": actual_freeze_sha,
    "final_official_test_record_sha256": actual_final_test_sha,
    "official_test_samples_explained": int(
        len(
            final_arrays[
                "sample_uid"
            ]
        )
    ),
    "methods": {
        "integrated_gradients": {
            "steps": 50,
            "target": "predicted-class logit",
            "ranking": "absolute IG on semantic tokens",
        },
        "cross_attention": "actual text CLS -> 196 patches; per-head and mean",
        "attention_rollout": "ViT rollout with residual identity",
        "incongruity": "internal ICAM a_j",
    },
    "training_performed": False,
    "threshold_calibration_performed": False,
    "test_based_model_or_sample_selection_performed": False,
    "next_action": "XAI_PERTURBATION",
}

STATUS_PATH = (
    OUTPUT_ROOT
    / "XAI_REPRESENTATIVE_C_STATUS.json"
)

STATUS_PATH.write_text(
    json.dumps(
        status,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        status,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "status": "PASS",
  "phase": "XAI_REPRESENTATIVE_C",
  "variant": "C",
  "representative_seed": 999,
  "representative_selection_source": "frozen validation-only rule",
  "representative_threshold": 0.45,
  "checkpoint_sha256": "fc76e1b4bf2288e99802356d63a8ad5345cef54732767c25c9e51178d21520c4",
  "protocol_sha256": "d3ba1de81ef89a67946c43af996396d294d35f01a1640168bab18c8a3bec4f12",
  "freeze_record_sha256": "63f8bba319f07928f3b0528bd3e8ebc3b7c325e77e3d3db01e7da6119ce31f62",
  "final_official_test_record_sha256": "c67b664e1ecb07bbbf0243550862e04ce3d2583518b18158c66c3ccc9fd5676b",
  "official_test_samples_explained": 1005,
  "methods": {
    "integrated_gradients": {
      "steps": 50,
      "target": "predicted-class logit",
      "ranking": "absolute IG on semantic tokens"
    },
    "cross_attention": "actual text CLS -> 196 patches; per-head and mean",
    "attention_rollout": "ViT rollout with residual identity",
    "incongruity": "internal ICAM a_j"
  },
  "training_performed"